# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract Summary

1. **Unit of Analysis (Grain):** One row = one unique content page (`content_id` / `page_path`) aggregated over a specific monthly window.
2. **Table(s) Used:** `mart_search_intelligence` (or `mart_content_performance` from DuckDB / Hugging Face warehouse).
3. **Time Window:** Mid-panel month: `2026-03` (March 1, 2026 to March 31, 2026).
4. **Target / Label (Proxy):** `is_high_value_decay` (Binary: 1 if `impressions_90d >= 1000`, `trend_direction == 'down'`, and `trend_pct <= -20%`). Continuous target: `opportunity_score_proxy` = $\text{impressions\_90d} \times \frac{|\text{trend\_pct}|}{100}$.
5. **Deliberately Excluded:** `post_refresh_clicks_30d` / future traffic metrics (excluded because they are unavailable at decision time and introduce target leakage).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field Name | Bucket | Reason / Description |
| :--- | :--- | :--- |
| `content_id` | **Context** | Identifier for tracking and joins |
| `publish_date` / `content_age_days` | **Feature** | Historical metadata, knowable at decision moment |
| `impressions_90d`, `clicks_90d` | **Feature** | Trailing 90-day activity leading up to mid-panel date |
| `trend_pct`, `trend_direction` | **Feature / Signal** | Historical performance trajectory |
| `is_high_value_decay` | **Label** | Target flag for identification of refresh candidates |
| `post_refresh_clicks_30d` | **Excluded** | Future outcome metric; causes data/target leakage |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
import duckdb
import pandas as pd

# 1. Connect DuckDB and register DataFrame 'df'
con = duckdb.connect()
con.register('df', df)

# --- Query 1: Grain Verification (Unique content_id per row) ---
query_grain = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_id) AS unique_content_ids,
    COUNT(*) = COUNT(DISTINCT content_id) AS is_grain_valid
FROM df;
"""
print("--- Query 1: Grain Check ---")
print(con.sql(query_grain).df())


# --- Query 2: Row Count & Content Age Span ---
query_counts = """
SELECT
    MIN(content_age_days) AS min_content_age_days,
    MAX(content_age_days) AS max_content_age_days,
    COUNT(*) AS total_rows
FROM df;
"""
print("\n--- Query 2: Age Span & Row Count ---")
print(con.sql(query_counts).df())


# --- Query 3: Availability Check (Completeness of metrics) ---
query_availability = """
SELECT
    COUNT(*) AS raw_count,
    COUNT(CASE WHEN impressions_90d IS NOT NULL AND clicks_90d IS NOT NULL THEN 1 END) AS valid_rows,
    ROUND(100.0 * COUNT(CASE WHEN impressions_90d IS NOT NULL THEN 1 END) / COUNT(*), 2) AS availability_pct
FROM df;
"""
print("\n--- Query 3: Availability Check ---")
print(con.sql(query_availability).df())

--- Query 1: Grain Check ---
   total_rows  unique_content_ids  is_grain_valid
0        4467                4467            True

--- Query 2: Age Span & Row Count ---
   min_content_age_days  max_content_age_days  total_rows
0                    90                   564        4467

--- Query 3: Availability Check ---
   raw_count  valid_rows  availability_pct
0       4467        4467             100.0


In [10]:
import duckdb
import pandas as pd

# 1. Clean non-numeric strings from trend_pct
df['trend_pct_num'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)
df['impressions_90d'] = pd.to_numeric(df['impressions_90d'], errors='coerce').fillna(0)
df['clicks_90d'] = pd.to_numeric(df['clicks_90d'], errors='coerce').fillna(0)

# Re-register cleaned df in DuckDB
con = duckdb.connect()
con.register('df', df)

# ---------------------------------------------------------
# 2. BUILD FEATURE FRAME (Max 5 Features + Rationale)
# ---------------------------------------------------------
df_features = con.sql("""
SELECT
    content_id,
    -- Feature 1: Historical Search Demand
    impressions_90d AS feat_impressions_90d,

    -- Feature 2: Historical Engagement (CTR)
    CASE WHEN impressions_90d > 0 THEN (clicks_90d * 1.0 / impressions_90d) ELSE 0 END AS feat_ctr_90d,

    -- Feature 3: Decay Severity (Historical drop magnitude using cleaned numeric column)
    ABS(LEAST(trend_pct_num, 0)) AS feat_decay_severity,

    -- Feature 4: Content Staleness
    content_age_days AS feat_content_age_days,

    -- Feature 5: Recency Signal
    days_since_last_update AS feat_days_since_update,

    -- Target Label
    CASE WHEN trend_direction = 'down' AND impressions_90d >= 1000 AND trend_pct_num <= -20 THEN 1 ELSE 0 END AS target_is_high_value_decay
FROM df
""").df()

print("--- Feature Frame Preview ---")
print(df_features.head())

# ---------------------------------------------------------
# 3. THE LEAKAGE TRAP (Deliberate Leakage Demonstration)
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Intentionally inject a feature derived directly from the label condition
df_features['LEAK_future_decay_flag'] = df_features['target_is_high_value_decay']

X_leaky = df_features[['feat_impressions_90d', 'feat_ctr_90d', 'feat_decay_severity', 'feat_content_age_days', 'feat_days_since_update', 'LEAK_future_decay_flag']]
y = df_features['target_is_high_value_decay']

clf = RandomForestClassifier(random_state=42)
clf.fit(X_leaky, y)
leaky_score = roc_auc_score(y, clf.predict_proba(X_leaky)[:, 1])

print(f"\n[LEAKED MODEL] ROC-AUC Score: {leaky_score:.4f} (Artificially perfect due to leakage!)")

# ---------------------------------------------------------
# 4. CLEAN HONEST BASELINE (Leakage Column Removed)
# ---------------------------------------------------------
X_clean = df_features[['feat_impressions_90d', 'feat_ctr_90d', 'feat_decay_severity', 'feat_content_age_days', 'feat_days_since_update']]
clf.fit(X_clean, y)
honest_score = roc_auc_score(y, clf.predict_proba(X_clean)[:, 1])

print(f"[HONEST MODEL] ROC-AUC Score: {honest_score:.4f} (Realistic evaluation baseline)")

--- Feature Frame Preview ---
             content_id  feat_impressions_90d  feat_ctr_90d  \
0  content_304f48230142                  3803      0.007626   
1  content_a1fb4e703a9e                 15320      0.000457   
2  content_9aa793d4d895                 12581      0.000874   
3  content_331d6c4de07b                 11751      0.004936   
4  content_d99b7a2d90ca                 19140      0.001254   

   feat_decay_severity  feat_content_age_days  feat_days_since_update  \
0                 41.4                    187                      20   
1                 57.7                    445                      25   
2                 60.9                    141                      20   
3                 13.8                    463                      22   
4                 34.7                    263                      14   

   target_is_high_value_decay  
0                           1  
1                           1  
2                           1  
3                       

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


### Data Limitations

* **Historical Unbalance:** Google Search Console (GSC) data is restricted to rolling window metrics. Older pieces of content lack detailed baseline performance logs from their initial launch period.
* **Aggregated Signals Only:** Early rows rely strictly on aggregated search impressions and clicks rather than rich session-level page analytics, limiting visibility into user scroll depth and conversions.
* **Window Overlap:** Evaluating content across rolling 90-day windows introduces smooth, overlapping temporal signals between consecutive evaluation windows rather than sharp, discrete performance shifts.